# SneakyToS — Dataset Reconnaissance and Zoom Pilot

This notebook performs the first data-engineering stage of the SneakyToS project.

## Goals

1. Inspect the Open Terms Archive `contrib-versions` repository.
2. Find services with tracked `Privacy Policy.md` files.
3. Extract Git metadata for every tracked Privacy Policy.
4. Build a dataset-level summary of historical policy coverage.
5. Save reconnaissance metadata for later notebooks.
6. Reconstruct all historical Zoom Privacy Policy versions as a pilot.
7. Detect exact duplicate policy snapshots using SHA-256 hashes.

The output of this notebook is metadata and a validated historical-version extraction pipeline. Semantic clause comparison and rights-expansion detection will be implemented in later notebooks.


## 1. Imports


In [1]:
from pathlib import Path
import subprocess
import hashlib
import pandas as pd


## 2. Project paths


In [2]:
# This notebook is expected to be inside:
# SneakyToS/notebooks/01_data_recon.ipynb

cwd = Path.cwd()

if (cwd / "data" / "raw" / "contrib-versions").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "raw" / "contrib-versions").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "data/raw/contrib-versions."
    )

DATASET_PATH = PROJECT_ROOT / "data" / "raw" / "contrib-versions"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATASET_PATH)
print("Dataset exists:", DATASET_PATH.exists())


Project root: c:\Users\arjun\Desktop\SneakyToS
Dataset path: c:\Users\arjun\Desktop\SneakyToS\data\raw\contrib-versions
Dataset exists: True


## 3. Inspect company folders


In [3]:
companies = sorted(
    [
        folder
        for folder in DATASET_PATH.iterdir()
        if folder.is_dir() and folder.name != ".git"
    ],
    key=lambda x: x.name.lower()
)

print("Number of company folders:", len(companies))
print("\nFirst 20 companies:")

for company in companies[:20]:
    print("-", company.name)


Number of company folders: 386

First 20 companies:
- 123Greetings
- Academia.edu
- ACDelco
- AcuityAds
- AdMob
- AdRoll
- AdSense
- AdSupply
- Aetna
- Airbnb
- Alexa
- Alibaba.com
- Allrecipes
- Allstate
- Alpha Vantage
- Amazon Advertising
- Amazon Appstore
- Amazon Coins
- Amazon Device
- Amazon E-Reader and Fire Tablet


## 4. Find tracked Privacy Policies


In [4]:
privacy_policies = []

for company in companies:
    policy_file = company / "Privacy Policy.md"

    if policy_file.exists():
        privacy_policies.append(
            {
                "company": company.name,
                "path": policy_file,
            }
        )

privacy_df = pd.DataFrame(privacy_policies)

print("Companies with Privacy Policies:", len(privacy_df))
privacy_df.head(10)


Companies with Privacy Policies: 307


,company,path
0,123Greetings,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
1,Academia.edu,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
2,ACDelco,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
3,AcuityAds,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
4,AdRoll,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
5,AdSense,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
6,AdSupply,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
7,Aetna,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
8,Airbnb,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...
9,Alibaba.com,c:\Users\arjun\Desktop\SneakyToS\data\raw\cont...


## 5. Git history extraction utilities


In [5]:
def get_git_history(repo_path, file_path):
    """Return commit hash, author timestamp, and message for a tracked file."""

    command = [
        "git",
        "-C",
        str(repo_path),
        "log",
        "--follow",
        "--format=%H|%aI|%s",
        "--",
        str(file_path),
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=True,
        encoding="utf-8",
        errors="replace",
    )

    return result.stdout


def parse_git_history(history_text, company):
    """Parse Git log output into structured records."""

    records = []

    for line in history_text.strip().split("\n"):
        parts = line.split("|", maxsplit=2)

        if len(parts) != 3:
            continue

        commit_hash, timestamp, message = parts

        records.append(
            {
                "company": company,
                "commit_hash": commit_hash.strip(),
                "timestamp": pd.to_datetime(
                    timestamp.strip(),
                    utc=True,
                ),
                "commit_message": message.strip(),
            }
        )

    return records


## 6. Sanity check: Zoom Git history


In [6]:
zoom_history_text = get_git_history(
    DATASET_PATH,
    Path("Zoom") / "Privacy Policy.md",
)

zoom_history_preview = pd.DataFrame(
    parse_git_history(
        zoom_history_text,
        "Zoom",
    )
).sort_values("timestamp").reset_index(drop=True)

print("Zoom tracked snapshots:", len(zoom_history_preview))
zoom_history_preview.head(10)


Zoom tracked snapshots: 30


,company,commit_hash,timestamp,commit_message
0,Zoom,f5197ca248b099f54512cb518181241af1642749,2021-08-16 08:55:23+00:00,Start tracking Zoom Privacy Policy
1,Zoom,4788e3698c43bc99597d7fd2836b138ef8145e85,2021-10-07 21:14:03+00:00,Update Zoom Privacy Policy
2,Zoom,4fbd6abb19ff42c3a828c95327248576a4966e0a,2021-11-03 05:10:18+00:00,Refilter Zoom Privacy Policy
3,Zoom,7669087fd9438400efc97cc939a9a061588e6da2,2021-11-23 10:17:24+00:00,Update Zoom Privacy Policy
4,Zoom,476756868644d8f9eeb6746d51418c1964c5ea4c,2022-01-13 18:43:08+00:00,Update Zoom Privacy Policy
5,Zoom,60927018fbf523614ab99e5cce0c701e75ca657b,2022-04-05 12:42:05+00:00,Update Zoom Privacy Policy
6,Zoom,5d41244ff973b31ad590f8a2c68159be1308ecae,2022-08-31 00:44:12+00:00,Update Zoom Privacy Policy
7,Zoom,961c91adeeb5859e9e0c99c4467ff17b4b9da348,2022-09-16 18:44:22+00:00,Update Zoom Privacy Policy
8,Zoom,0c0014d9036d2bed3f4832efb4206a6063c63fc0,2022-09-20 15:23:57+00:00,Refilter Zoom Privacy Policy
9,Zoom,3fda78b39fabedb62d68a7957d4a60ec085c89fd,2022-11-07 12:46:22+00:00,Update Zoom Privacy Policy


## 7. Extract metadata for all Privacy Policies


In [7]:
all_records = []
failed_companies = []

for i, row in privacy_df.iterrows():
    company = row["company"]
    relative_path = Path(company) / "Privacy Policy.md"

    try:
        history = get_git_history(
            DATASET_PATH,
            relative_path,
        )

        records = parse_git_history(
            history,
            company,
        )

        all_records.extend(records)

    except subprocess.CalledProcessError as e:
        failed_companies.append(
            {
                "company": company,
                "error": str(e),
            }
        )

    if (i + 1) % 25 == 0:
        print(
            f"Processed {i + 1}/{len(privacy_df)} companies"
        )

print("\nTotal version records:", len(all_records))
print("Failed companies:", len(failed_companies))


Processed 25/307 companies
Processed 50/307 companies
Processed 75/307 companies
Processed 100/307 companies
Processed 125/307 companies
Processed 150/307 companies
Processed 175/307 companies
Processed 200/307 companies
Processed 225/307 companies
Processed 250/307 companies
Processed 275/307 companies
Processed 300/307 companies

Total version records: 18336
Failed companies: 0


## 8. Build the complete policy-history table


In [8]:
history_df = pd.DataFrame(all_records)

history_df = (
    history_df
    .sort_values(
        ["company", "timestamp"]
    )
    .reset_index(drop=True)
)

print(history_df.info())
history_df.head()


<class 'pandas.DataFrame'>
RangeIndex: 18336 entries, 0 to 18335
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   company         18336 non-null  str                
 1   commit_hash     18336 non-null  str                
 2   timestamp       18336 non-null  datetime64[us, UTC]
 3   commit_message  18336 non-null  str                
dtypes: datetime64[us, UTC](1), str(3)
memory usage: 573.1 KB
None


,company,commit_hash,timestamp,commit_message
0,123Greetings,0a37d41432355d380bd486f72ab1258c78987063,2012-10-03 20:16:21+00:00,Start tracking 123Greetings Privacy Policy
1,123Greetings,978cb6a43ec8874fb73b77e1b76ac342091c97e3,2012-10-10 21:18:01+00:00,Update 123Greetings Privacy Policy
2,123Greetings,a74fa57b4b08ab4867819af11a2c94b62215376a,2018-05-10 05:43:56+00:00,Update 123Greetings Privacy Policy
3,123Greetings,f31af92915326aaa76e56d87921edb0fe0c3a6f8,2018-05-19 05:45:06+00:00,Update 123Greetings Privacy Policy
4,123Greetings,043f3ea4b20d21cf7472ac7e0fa6faa8a5a4ee95,2018-05-20 05:48:46+00:00,Update 123Greetings Privacy Policy


## 9. Dataset-level summary


In [9]:
summary_df = (
    history_df
    .groupby("company")
    .agg(
        versions=("commit_hash", "count"),
        first_version=("timestamp", "min"),
        last_version=("timestamp", "max"),
    )
    .reset_index()
)

summary_df["consecutive_pairs"] = (
    summary_df["versions"] - 1
)

summary_df = (
    summary_df
    .sort_values(
        by="versions",
        ascending=False,
    )
    .reset_index(drop=True)
)

summary_df.head(20)


,company,versions,first_version,last_version,consecutive_pairs
0,Xanga,2484,2012-10-04 11:25:40+00:00,2025-09-23 06:30:20+00:00,2483
1,Alibaba.com,971,2012-10-04 09:17:23+00:00,2026-08-20 12:32:03+00:00,970
2,Allstate,922,2012-10-04 09:17:23+00:00,2026-08-06 12:32:55+00:00,921
3,Windstream,877,2020-11-09 17:31:56+00:00,2023-01-14 06:45:50+00:00,876
4,Zillow,510,2012-10-04 11:25:40+00:00,2025-11-15 06:30:35+00:00,509
5,Booking.com,492,2020-10-20 15:30:12+00:00,2026-08-20 12:35:17+00:00,491
6,Disqus,484,2020-12-08 15:20:25+00:00,2026-08-10 00:37:23+00:00,483
7,CommentCaMarche,434,2020-12-08 15:20:13+00:00,2026-08-28 00:36:52+00:00,433
8,Wikimedia Foundation,384,2013-06-20 08:50:34+00:00,2026-09-02 13:06:11+00:00,383
9,Quora,331,2020-07-29 13:32:34+00:00,2025-09-23 07:41:11+00:00,330


## 10. Basic dataset statistics


In [10]:
print(
    "Companies with Privacy Policy history:",
    len(summary_df),
)

print(
    "Total policy versions:",
    summary_df["versions"].sum(),
)

print(
    "Total possible consecutive version pairs:",
    summary_df["consecutive_pairs"].sum(),
)

print("\nVersion statistics:")
print(
    summary_df["versions"].describe()
)


Companies with Privacy Policy history: 307
Total policy versions: 18336
Total possible consecutive version pairs: 18029

Version statistics:
count     307.000000
mean       59.726384
std       182.600308
min         1.000000
25%         6.000000
50%        14.000000
75%        35.000000
max      2484.000000
Name: versions, dtype: float64


## 11. Candidate-company exploration


In [11]:
candidates_10_plus = (
    summary_df[
        summary_df["versions"] >= 10
    ]
    .copy()
)

candidates_5_to_9 = (
    summary_df[
        (summary_df["versions"] >= 5)
        & (summary_df["versions"] < 10)
    ]
    .copy()
)

print(
    "Companies with at least 10 versions:",
    len(candidates_10_plus),
)

print(
    "Companies with 5–9 versions:",
    len(candidates_5_to_9),
)

print("\nTop 30 companies by number of tracked snapshots:")
display(summary_df.head(30))


Companies with at least 10 versions: 194
Companies with 5–9 versions: 58

Top 30 companies by number of tracked snapshots:


,company,versions,first_version,last_version,consecutive_pairs
0,Xanga,2484,2012-10-04 11:25:40+00:00,2025-09-23 06:30:20+00:00,2483
1,Alibaba.com,971,2012-10-04 09:17:23+00:00,2026-08-20 12:32:03+00:00,970
2,Allstate,922,2012-10-04 09:17:23+00:00,2026-08-06 12:32:55+00:00,921
3,Windstream,877,2020-11-09 17:31:56+00:00,2023-01-14 06:45:50+00:00,876
4,Zillow,510,2012-10-04 11:25:40+00:00,2025-11-15 06:30:35+00:00,509
5,Booking.com,492,2020-10-20 15:30:12+00:00,2026-08-20 12:35:17+00:00,491
6,Disqus,484,2020-12-08 15:20:25+00:00,2026-08-10 00:37:23+00:00,483
7,CommentCaMarche,434,2020-12-08 15:20:13+00:00,2026-08-28 00:36:52+00:00,433
8,Wikimedia Foundation,384,2013-06-20 08:50:34+00:00,2026-09-02 13:06:11+00:00,383
9,Quora,331,2020-07-29 13:32:34+00:00,2025-09-23 07:41:11+00:00,330


## 12. Save reconnaissance metadata


In [12]:
summary_output_path = (
    PROCESSED_PATH
    / "policy_history_summary.csv"
)

history_output_path = (
    PROCESSED_PATH
    / "policy_version_history.csv"
)

summary_df.to_csv(
    summary_output_path,
    index=False,
)

history_df.to_csv(
    history_output_path,
    index=False,
)

print("Saved:", summary_output_path)
print("Saved:", history_output_path)


Saved: c:\Users\arjun\Desktop\SneakyToS\data\processed\policy_history_summary.csv
Saved: c:\Users\arjun\Desktop\SneakyToS\data\processed\policy_version_history.csv


# Zoom Historical-Version Pilot

The following cells validate that historical policy text can be reconstructed from Git commits and allow us to identify exact duplicate snapshots.


## 13. Historical file reconstruction utility


In [13]:
def get_file_at_commit(
    repo_path,
    commit_hash,
    relative_path,
):
    """Return the contents of a file at a specific Git commit."""

    command = [
        "git",
        "-C",
        str(repo_path),
        "show",
        f"{commit_hash}:{relative_path.as_posix()}",
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=True,
        encoding="utf-8",
        errors="replace",
    )

    return result.stdout


def get_text_hash(text):
    """Return a SHA-256 hash for exact-content deduplication."""

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


## 14. Select Zoom versions in exact chronological order


In [14]:
zoom_history = (
    history_df[
        history_df["company"] == "Zoom"
    ]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(
    f"Zoom versions to extract: {len(zoom_history)}"
)

zoom_history[
    [
        "timestamp",
        "commit_hash",
        "commit_message",
    ]
]


Zoom versions to extract: 30


,timestamp,commit_hash,commit_message
0,2021-08-16 08:55:23+00:00,f5197ca248b099f54512cb518181241af1642749,Start tracking Zoom Privacy Policy
1,2021-10-07 21:14:03+00:00,4788e3698c43bc99597d7fd2836b138ef8145e85,Update Zoom Privacy Policy
2,2021-11-03 05:10:18+00:00,4fbd6abb19ff42c3a828c95327248576a4966e0a,Refilter Zoom Privacy Policy
3,2021-11-23 10:17:24+00:00,7669087fd9438400efc97cc939a9a061588e6da2,Update Zoom Privacy Policy
4,2022-01-13 18:43:08+00:00,476756868644d8f9eeb6746d51418c1964c5ea4c,Update Zoom Privacy Policy
5,2022-04-05 12:42:05+00:00,60927018fbf523614ab99e5cce0c701e75ca657b,Update Zoom Privacy Policy
6,2022-08-31 00:44:12+00:00,5d41244ff973b31ad590f8a2c68159be1308ecae,Update Zoom Privacy Policy
7,2022-09-16 18:44:22+00:00,961c91adeeb5859e9e0c99c4467ff17b4b9da348,Update Zoom Privacy Policy
8,2022-09-20 15:23:57+00:00,0c0014d9036d2bed3f4832efb4206a6063c63fc0,Refilter Zoom Privacy Policy
9,2022-11-07 12:46:22+00:00,3fda78b39fabedb62d68a7957d4a60ec085c89fd,Update Zoom Privacy Policy


## 15. Extract all historical Zoom Privacy Policy texts


In [15]:
zoom_version_records = []

for i, row in zoom_history.iterrows():

    company = row["company"]
    commit_hash = row["commit_hash"]

    relative_path = (
        Path(company)
        / "Privacy Policy.md"
    )

    try:
        text = get_file_at_commit(
            DATASET_PATH,
            commit_hash,
            relative_path,
        )

        zoom_version_records.append(
            {
                "company": company,
                "commit_hash": commit_hash,
                "timestamp": row["timestamp"],
                "commit_message": row["commit_message"],
                "text": text,
                "text_hash": get_text_hash(text),
                "char_count": len(text),
                "word_count": len(text.split()),
            }
        )

        print(
            f"[{i + 1}/{len(zoom_history)}] "
            f"{row['timestamp']} | "
            f"{len(text.split())} words"
        )

    except Exception as e:
        print(
            f"FAILED: {company} | {commit_hash}"
        )
        print(e)

print(
    f"\nSuccessfully extracted: "
    f"{len(zoom_version_records)} versions"
)


[1/30] 2021-08-16 08:55:23+00:00 | 4679 words
[2/30] 2021-10-07 21:14:03+00:00 | 4705 words
[3/30] 2021-11-03 05:10:18+00:00 | 4829 words
[4/30] 2021-11-23 10:17:24+00:00 | 4829 words
[5/30] 2022-01-13 18:43:08+00:00 | 4774 words
[6/30] 2022-04-05 12:42:05+00:00 | 4774 words
[7/30] 2022-08-31 00:44:12+00:00 | 4775 words
[8/30] 2022-09-16 18:44:22+00:00 | 5361 words
[9/30] 2022-09-20 15:23:57+00:00 | 5358 words
[10/30] 2022-11-07 12:46:22+00:00 | 6080 words
[11/30] 2022-11-08 18:44:48+00:00 | 6125 words
[12/30] 2022-11-10 00:44:41+00:00 | 6126 words
[13/30] 2022-11-19 00:44:15+00:00 | 6126 words
[14/30] 2022-12-30 12:46:18+00:00 | 7000 words
[15/30] 2022-12-31 00:45:55+00:00 | 6996 words
[16/30] 2023-01-19 06:39:23+00:00 | 6126 words
[17/30] 2023-01-19 06:55:06+00:00 | 6996 words
[18/30] 2023-02-24 18:46:12+00:00 | 7008 words
[19/30] 2023-04-10 18:45:00+00:00 | 7143 words
[20/30] 2023-06-30 12:48:00+00:00 | 7183 words
[21/30] 2023-08-11 19:15:46+00:00 | 7367 words
[22/30] 2023-09-05 12:

## 16. Create the Zoom version table


In [16]:
zoom_versions_df = (
    pd.DataFrame(zoom_version_records)
    .sort_values("timestamp")
    .reset_index(drop=True)
)

zoom_versions_df[
    [
        "timestamp",
        "commit_hash",
        "commit_message",
        "word_count",
        "char_count",
        "text_hash",
    ]
]


,timestamp,commit_hash,commit_message,word_count,char_count,text_hash
0,2021-08-16 08:55:23+00:00,f5197ca248b099f54512cb518181241af1642749,Start tracking Zoom Privacy Policy,4679,31458,4b2e66fe8b0ba75a7d2510623c618fba549d2fde360080...
1,2021-10-07 21:14:03+00:00,4788e3698c43bc99597d7fd2836b138ef8145e85,Update Zoom Privacy Policy,4705,31710,330b2a9d5e468d54cec04499bcf135d0f34ce223923db0...
2,2021-11-03 05:10:18+00:00,4fbd6abb19ff42c3a828c95327248576a4966e0a,Refilter Zoom Privacy Policy,4829,32622,1c3199181727797b4b323da4d20c197d3175d80ac55985...
3,2021-11-23 10:17:24+00:00,7669087fd9438400efc97cc939a9a061588e6da2,Update Zoom Privacy Policy,4829,32630,f333569a1fc99346c8b2ebe8ce130aaf06eb8df3dccafd...
4,2022-01-13 18:43:08+00:00,476756868644d8f9eeb6746d51418c1964c5ea4c,Update Zoom Privacy Policy,4774,32748,a3470c3441ac72dcfb759bfb8cb547ea92ad2a69b9e7b0...
5,2022-04-05 12:42:05+00:00,60927018fbf523614ab99e5cce0c701e75ca657b,Update Zoom Privacy Policy,4774,32748,d392a862224f89866e88e283f8eb50fa1b076af4f4839a...
6,2022-08-31 00:44:12+00:00,5d41244ff973b31ad590f8a2c68159be1308ecae,Update Zoom Privacy Policy,4775,32753,193561e5e2492418fcb43cd389aa9c7c73552f6fa98179...
7,2022-09-16 18:44:22+00:00,961c91adeeb5859e9e0c99c4467ff17b4b9da348,Update Zoom Privacy Policy,5361,36504,e1818ff782b5e716de749b2dbf91c29a7ea7978105be37...
8,2022-09-20 15:23:57+00:00,0c0014d9036d2bed3f4832efb4206a6063c63fc0,Refilter Zoom Privacy Policy,5358,36369,f7fae2bd2f59c7dade098973e6f545b250e0c2faab9972...
9,2022-11-07 12:46:22+00:00,3fda78b39fabedb62d68a7957d4a60ec085c89fd,Update Zoom Privacy Policy,6080,40957,86aaa00cc0b4bc81d6310ddc5649f60a281a1ca83d0497...


## 17. Exact duplicate analysis


In [17]:
git_snapshots = len(zoom_versions_df)

unique_policy_texts = (
    zoom_versions_df["text_hash"].nunique()
)

exact_duplicate_snapshots = (
    git_snapshots
    - unique_policy_texts
)

print("Git snapshots:", git_snapshots)
print(
    "Unique policy texts:",
    unique_policy_texts,
)
print(
    "Exact duplicate snapshots:",
    exact_duplicate_snapshots,
)


Git snapshots: 30
Unique policy texts: 28
Exact duplicate snapshots: 2


## 18. Inspect exact duplicate snapshots, if any


In [18]:
duplicate_versions = (
    zoom_versions_df[
        zoom_versions_df.duplicated(
            subset="text_hash",
            keep=False,
        )
    ]
    .sort_values(
        ["text_hash", "timestamp"]
    )
)

if duplicate_versions.empty:
    print(
        "No exact duplicate Zoom policy texts were found."
    )
else:
    display(
        duplicate_versions[
            [
                "timestamp",
                "commit_hash",
                "commit_message",
                "word_count",
                "text_hash",
            ]
        ]
    )


,timestamp,commit_hash,commit_message,word_count,text_hash
12,2022-11-19 00:44:15+00:00,48fbc537524305644c9036d37f0eb258ed673f85,Update Zoom Privacy Policy,6126,68aebf329cab657d01473e6f1e44b497c0b4da73de6ecd...
15,2023-01-19 06:39:23+00:00,0d97e95b2d0e86853926ce5b694d39f86c905637,Refilter Zoom Privacy Policy,6126,68aebf329cab657d01473e6f1e44b497c0b4da73de6ecd...
14,2022-12-31 00:45:55+00:00,5ffee27416ab3a78e2d3853a8238d191456f3638,Update Zoom Privacy Policy,6996,a7ae050e961dfcd2933d433056a4588cd4c641542fe093...
16,2023-01-19 06:55:06+00:00,7e19aca26bc91dfaf447f8033b6f15375cb7e36a,Update Zoom Privacy Policy,6996,a7ae050e961dfcd2933d433056a4588cd4c641542fe093...


## 19. Save the Zoom pilot dataset


In [19]:
zoom_output_path = (
    PROCESSED_PATH
    / "zoom_policy_versions.pkl"
)

zoom_versions_df.to_pickle(
    zoom_output_path
)

print(
    f"Saved Zoom pilot dataset to: "
    f"{zoom_output_path}"
)


Saved Zoom pilot dataset to: c:\Users\arjun\Desktop\SneakyToS\data\processed\zoom_policy_versions.pkl


# Notebook Output

After successful execution, this notebook produces:

- `data/processed/policy_history_summary.csv`
- `data/processed/policy_version_history.csv`
- `data/processed/zoom_policy_versions.pkl`

## Next Notebook

`02_version_pairs_and_baselines.ipynb`

The next stage will:

1. Remove or collapse exact duplicate policy snapshots.
2. Create chronologically consecutive unique-version pairs.
3. Compute surface text-diff baselines.
4. Measure change magnitude.
5. Prepare version pairs for clause alignment and semantic rights-scope analysis.
